# 10 工作流与可视化平台 · 01 可视化平台与 Langflow

上一章我们一直在**用代码编排工作流**（`StateGraph` / 条件边 / 子图）。
这一课换一个视角问同一个问题：**能不能不写代码，把工作流「拖」出来？**

答案是可以，而且有四条不同的路。本节把四条路摆在一起对比，然后**落回代码**——
因为可视化平台搭完流程之后的下一步，永远是把流程**当接口调**。

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 可视化平台 | 用拖节点代替写代码的编排工具 | Coze / Dify / n8n / Langflow 四选一 |
| 平台 vs 框架 | 平台给你界面+运行时，框架给你 API | Dify ↔ LangChain、Langflow ↔ LangGraph |
| flow（流程） | 平台上搭出来的一条工作流，有唯一 ID | `flow_id`，出现在 URL 的 `/flow/<这一段>` |
| 流程即接口 | 把 flow 暴露成 HTTP，外部系统直接 POST | `POST /api/v1/run/<flow_id>` |
| payload 四字段 | 调这个接口要传的四样东西 | `output_type` / `input_type` / `input_value` / `session_id` |
| 健康探测 | 发请求前先问一句「服务在吗」 | `GET {base_url}/health` |
| 失败降级 | 服务不在时打印中文提示，而不是抛异常 | `--dry-run` 分支 |

> **本 notebook 由 `Agent/10_workflow_platform/` 下 2 个脚本合并而成**：
> `01_langflow_api_jxsd.py`（完整版 394 行）、`02_平台对比与部署_jxsd.py`（完整版 430 行）。
> 两个文件互为前后脚：**02 讲怎么选、怎么装；01 讲装好之后怎么调**。
> 本节按「先选型（02）→ 后调用（01）」的顺序串成一条链路。

**官方文档**
- 工作流与 Agent 的官方说明（本节的“代码编排”那一半）：<https://docs.langchain.com/oss/python/langgraph/workflows-agents>
- Langflow 官方文档：<https://docs.langflow.org/>
- Dify 官方文档：<https://docs.dify.ai/>
- n8n 官方文档：<https://docs.n8n.io/>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟢 运行档位 | **离线可跑** —— 不读 `.env`、不连大模型、不起任何服务 |
| 依赖 | `requests`（本项目 venv 已装 2.34.2），其余全是标准库 |
| 密钥 | 不需要。全程只用占位符 `YOUR_LANGFLOW_API_KEY` / `YOUR_FLOW_ID` |
| 前置服务 | **无**。Langflow 没启动时，本 notebook 走源文件自带的中文降级路径 |
| 预计耗时 | 约 20 秒（其中内核启动占大头；真正执行的部分不到 2 秒） |

本 notebook 里有**三种不同颜色的运行档位**，请分开看：

| 段落 | 档位 | 需要什么 |
|---|---|---|
| 第 1、2 节（平台对比 + 五段部署命令） | 🟢 离线 | 什么都不需要 |
| 第 2.7 节（本机环境探测） | 🟢 离线 | 有个 `docker` 命令更好；没有也能跑，只是打印安装指引 |
| 第 3.8 节（**真实调用** Langflow flow） | 🔴 需外部服务 | 已启动的 Langflow + `LANGFLOW_URL`/`LANGFLOW_BASE_URL`、`LANGFLOW_FLOW_ID`、`LANGFLOW_API_KEY` |

> 第 3 节 3.1~3.7 全部走**降级路径**：先 `GET /health` 探测，连不上就打印
> 「将要发送的请求长什么样」+ 一段中文排障指引，**不抛异常**。
> 这是源文件 `01_langflow_api_jxsd.py` 里就写好的分支，不是本节改的。

## 本节地图

先看这一课的位置：它接在「代码编排工作流」之后，回答「要不要换成可视化平台」，
再把两条路合流到「流程即接口」。

```mermaid
graph LR
    A["代码编排<br/>LangGraph StateGraph<br/>（上一章）"] --> B{"要不要<br/>可视化平台？"}
    B -->|"不写代码给同事用"| C["Coze<br/>闭源 SaaS"]
    B -->|"上线 RAG / Agent"| D["Dify<br/>开源 · 面向开发者"]
    B -->|"跨系统自动化"| E["n8n<br/>开源 · 面向专业开发者"]
    B -->|"想可视化调 LangGraph"| F["Langflow<br/>开源 · 暴露 LangChain 组件"]
    F --> G["搭一条 flow<br/>拿到 flow_id"]
    G --> H["POST /api/v1/run/flow_id<br/>流程即接口"]
    D --> I["Dify 工作流<br/>（同样可发布成 API）"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 从 | 到 | 什么时候走 |
|---|---|---|
| 代码编排（LangGraph） | 「要不要可视化平台」这个判断 | 本节开头 |
| 判断 | Coze | 只想拖几个节点做个 bot 给同事用 |
| 判断 | Dify | 要做能上线的 RAG / Agent 应用，团队有后端 |
| 判断 | n8n | 要把 100 个系统串起来做自动化 |
| 判断 | Langflow | 已经在用 LangChain/LangGraph，想要个可视化调试界面 |
| Langflow | 搭一条 flow → 拿 `flow_id` | 拖完节点、点保存之后 |
| `flow_id` | `POST /api/v1/run/<flow_id>` | **本 notebook 第 3 节**：把流程当接口调 |

**这一课的落点**：平台只是壳，真正跑起来的还是工作流本身；
而「把它当接口调」这件事，无论用哪个平台，形状都是
`POST + 几个字段 + 一个会话 ID`。第 3 节就是把这四个字段讲透。

> 与上一章 `01_langgraph/*` 的衔接：上一章是**自己写**图，
> 这一节是**用别人做好的界面**画同一张图，最后再用 HTTP 把它当服务调起来。
> 下一章 `11` 之后会回到「自己写」，但那时你会知道什么时候该偷懒用平台。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

> 本课其实用不到 `config`（不读任何密钥），但这一格仍然保留 —— 一是全仓统一，
> 二是它顺便给出 `NB_DIR` / `WORKDIR` 两个变量，将来本课要落盘时直接用。

> ⚠️ 源文件开头的 `sys.stdout.reconfigure(encoding="utf-8")` 在这一格里**不写**：
> 那行是为了救 Windows 控制台的 GBK，而 Jupyter 内核的输出流本来就是 UTF-8，
> 硬写会 `AttributeError`（`OutStream` 没有 `reconfigure`）。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

```text
仓库根： F:\ProGram\Python_Base
临时目录： F:\ProGram\Python_Base\Agent\10_workflow_platform\tmp_nb_work
```

## 0.1 前置条件自检

本课是 🟢 离线档，**理论上什么都不缺也能跑完**。这一格只做一件事：
把「本机到底有什么」先打出来，这样后面哪一段走了降级路径，你一眼就知道为什么。

检查四样东西：

| 检查项 | 缺了会怎样 |
|---|---|
| `requests` | 第 3 节整节跳过（本机 venv 已装，一般不会缺） |
| `LANGFLOW_*` 环境变量 | 不影响运行：会用占位符，只是真实调用一定失败 |
| `docker` 命令 | 不影响运行：第 2.7 节会打印安装指引 |
| `127.0.0.1:7860` 是否在监听 | 不影响运行：第 3 节走 dry-run 降级分支 |

In [ ]:
import shutil
import socket

_todo: list[str] = []

try:
    import requests

    print(f"[OK] requests {requests.__version__} 已装 —— 第 3 节的 HTTP 调用可用")
except ImportError:
    _todo.append("requests")
    print("[跳过] 缺 requests：第 3 节整节跳过（本机 venv 已装，一般不会走到这里）")

_base_url = os.getenv("LANGFLOW_BASE_URL", "http://localhost:7860")
print(f"[信息] LANGFLOW_BASE_URL = {_base_url}（未设置时默认 http://localhost:7860）")
print(
    "[信息] LANGFLOW_API_KEY  = "
    + ("已设置" if os.getenv("LANGFLOW_API_KEY") else "未设置（第 3 节会用占位符 YOUR_LANGFLOW_API_KEY）")
)
print(
    "[信息] LANGFLOW_FLOW_ID  = "
    + ("已设置" if os.getenv("LANGFLOW_FLOW_ID") else "未设置（第 3 节会用占位符 YOUR_FLOW_ID）")
)

_docker = shutil.which("docker")
print(f"[信息] docker 命令：{_docker or '未安装 —— 第 2.7 节会打印安装指引，不报错'}")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
    sock.settimeout(0.5)
    _listening = sock.connect_ex(("127.0.0.1", 7860)) == 0
print(f"[信息] 127.0.0.1:7860：{'已在监听（Langflow 可能正在运行）' if _listening else '无服务监听 —— 第 3 节会走 dry-run 降级路径'}")

if _todo:
    print(f"[跳过] 缺少：{_todo} —— 第 3 节会被跳过")

### 预期输出

本机（Windows + 该项目 `.venv`）实测输出：

```text
[OK] requests 2.34.2 已装 —— 第 3 节的 HTTP 调用可用
[信息] LANGFLOW_BASE_URL = http://localhost:7860（未设置时默认 http://localhost:7860）
[信息] LANGFLOW_API_KEY  = 未设置（第 3 节会用占位符 YOUR_LANGFLOW_API_KEY）
[信息] LANGFLOW_FLOW_ID  = 未设置（第 3 节会用占位符 YOUR_FLOW_ID）
[信息] docker 命令：C:\Program Files\Docker\Docker\resources\bin\docker.EXE
[信息] 127.0.0.1:7860：无服务监听 —— 第 3 节会走 dry-run 降级路径
```

两处会随机器变化，不必对号入座：

- `docker 命令` 的路径（没装 Docker 时这一行会打印中文安装指引）；
- `7860` 那一行（真起了 Langflow 就会变成「已在监听」）。

## 1. 课案原文：那张对比表 + 那段 API 调用代码

本课的课案给了两样东西，正好是一头一尾：

1. **一张四平台对比表**（「对比」那节）—— 决定「用哪个」；
2. **一段 Langflow 生成的 API 调用代码** —— 决定「怎么调」。

先看 1.1 的表（这是 notebook 化收益最大的地方：课案里它是注释里的一张 ASCII 表，
现在它是**真的 Markdown 表格**，能排序、能加列）；1.2 再看那段代码。

### 1.1 四平台对比表（课案原文 + 本节补充两列）

| 平台 | 定位（课案原文） | 开源 | 部署形态 | 强项 | 什么时候选它 |
|---|---|---|---|---|---|
| **Coze** | 面向普通用户，适合日常使用 | ❌ 否 | 云端 SaaS | 零门槛搭 bot | 不想写代码，只想拖几个节点做个 bot 给同事用 |
| **Dify** | 面向开发者 | ✅ 是 | Docker Compose 私有化 | RAG 链路开箱即用（分段 / 检索 / 重排 / 引用） | 要做能上线的 RAG / Agent 应用，团队有后端 |
| **n8n** | 面向专业开发者 | ✅ 是 | Docker / npm | 几百个现成集成节点 | 要把 100 个系统串起来做自动化（GitHub、飞书、数据库） |
| **Langflow** | 可视化 LangChain/LangGraph，面向专业开发者，适合定制化开发 | ✅ 是 | conda（开发）/ docker compose（生产） | 直接暴露 LangChain 组件，能细调到代码级 | 已经在用 LangChain/LangGraph，想要个可视化调试界面 |

地址：Coze <https://code.coze.cn/home> ，Dify <https://cloud.dify.ai/>

**课案表格之外的两条取舍依据**（源文件里有、课案没展开）：

- **开源与否决定的不是「免不免费」，而是「能不能私有化部署、能不能改源码」。**
  Coze 闭源 → 数据必须过它的云；Dify / n8n / Langflow 都能整包丢进内网。
- **「面向开发者」的差别在抽象层次**：Dify 把 RAG 链路做成了开箱即用的节点；
  n8n 把「集成」做成了几百个现成节点；Langflow 直接暴露 LangChain/LangGraph 的组件，
  能细调到代码级，**代价是你要懂 LangChain 的概念**（Chain / Retriever / Tool）。
- 本仓库 `Agent/` 目录下那些 LangGraph 代码，配一个 Langflow 界面调试是最顺的；
  如果只是想把「每天抓 GitHub 热榜发到群里」跑起来，n8n 更省事。

### 1.2 课案那段代码：`POST /api/v1/run/<flow_id>`

课案在 Langflow 里拖出一个 RAG 问答流程后，生成了访问密钥，给出了这段调用代码。
下面的 `COURSE_CODE` 就是**课案原文**（逐字保留，`api_key` 处本来写的就是占位符
`'YOUR_API_KEY'`，本文件没有做任何替换）。

两个占位符常量放在同一格里，是为了让「代码里没有真实密钥」这件事**可被搜索验证**：
在全仓库搜 `YOUR_LANGFLOW_API_KEY`，命中的只可能是占位符。

In [ ]:
import argparse
import json
import uuid

# requests 不是标准库，但它是本节的必需品（课案那段也是用它）。
# 用 try/except 包住是为了「缺包时给一句人话」，而不是让 ImportError 糊在屏幕上 ——
# 教学脚本的失败提示本身也是教学内容。
try:
    import requests
except ImportError:  # pragma: no cover —— 本机已装（requests 2.34.2），这里只是保底
    print("缺少依赖 requests，请执行：uv add requests")
    sys.exit(0)


# 课案原文（api_key 处已用占位符；课案写的就是 'YOUR_API_KEY'）
COURSE_CODE = '''import requests
import os
import uuid


api_key = 'YOUR_API_KEY'
url = "http://localhost:7860/api/v1/run/你的flow_id"  # The complete API endpoint URL for this flow


# Request payload configuration
payload = {
    "output_type": "chat",
    "input_type": "chat",
    "input_value": "橘醒时代"
}
payload["session_id"] = str(uuid.uuid4())


headers = {"x-api-key": api_key}


try:
    # Send API request
    response = requests.request("POST", url, json=payload, headers=headers)
    response.raise_for_status()  # Raise exception for bad status codes


    # Print response
    print(response.text)


except requests.exceptions.RequestException as e:
    print(f"Error making API request: {e}")
except ValueError as e:
    print(f"Error parsing response: {e}")
'''

# 占位符常量：搜索这两个字符串就能确认「代码里没有真实密钥」
PLACEHOLDER_API_KEY = "YOUR_LANGFLOW_API_KEY"
PLACEHOLDER_FLOW_ID = "YOUR_FLOW_ID"

课案这段代码本身没毛病，但**直接抄进项目会踩三个坑**：

| # | 坑 | 现象 | 本节怎么补 |
|---|---|---|---|
| 1 | `api_key` 硬编码在源码里 | 一旦提交进 Git 就等于泄露 | 参数外置：命令行 > 环境变量 > 占位符 |
| 2 | `flow_id` 硬编码 | 开发和线上是两个流程 ID，改不动 | 同上（`--flow-id` / `LANGFLOW_FLOW_ID`） |
| 3 | `requests.request` 会抛 `ConnectionError` | Langflow 没启动时直接崩，不给提示 | 发送前先 `GET /health` 探测，失败降级为 dry-run |

下面这一格就是把这三点讲出来（`section_course()` 是源文件里的「课案原文」小节）。

In [ ]:
def section_course() -> None:
    print("=" * 78)
    print("1. 课案原文：Langflow 生成的 API 调用代码")
    print("=" * 78)
    print(COURSE_CODE)
    print("-" * 78)
    print(
        f"""
课案这段代码本身没毛病，但直接抄进项目会踩三个坑：

    坑 1：api_key 硬编码在源码里。真实项目里密钥必须走环境变量，
          否则一旦提交进 Git 就等于泄露（本文件用占位符 {PLACEHOLDER_API_KEY}）。
    坑 2：flow_id 硬编码。开发和线上用的是两个流程 ID，必须能配。
    坑 3：requests.request 会抛 ConnectionError。Langflow 没启动时，
          课案那段会直接崩，而不是给出「请先启动服务」的提示。

本文件把这三个坑都补上：参数外置 + 健康探测 + 失败降级。
"""
    )


section_course()

### 预期输出

```text
==============================================================================
1. 课案原文：Langflow 生成的 API 调用代码
==============================================================================
import requests
import os
import uuid


api_key = 'YOUR_API_KEY'
url = "http://localhost:7860/api/v1/run/你的flow_id"  # The complete API endpoint URL for this flow


# Request payload configuration
payload = {
    "output_type": "chat",
    "input_type": "chat",
    "input_value": "橘醒时代"
}
payload["session_id"] = str(uuid.uuid4())


headers = {"x-api-key": api_key}


try:
    # Send API request
    response = requests.request("POST", url, json=payload, headers=headers)
    response.raise_for_status()  # Raise exception for bad status codes


    # Print response
    print(response.text)


except requests.exceptions.RequestException as e:
    print(f"Error making API request: {e}")
except ValueError as e:
    print(f"Error parsing response: {e}")

------------------------------------------------------------------------------

课案这段代码本身没毛病，但直接抄进项目会踩三个坑：

    坑 1：api_key 硬编码在源码里。真实项目里密钥必须走环境变量，
          否则一旦提交进 Git 就等于泄露（本文件用占位符 YOUR_LANGFLOW_API_KEY）。
    坑 2：flow_id 硬编码。开发和线上用的是两个流程 ID，必须能配。
    坑 3：requests.request 会抛 ConnectionError。Langflow 没启动时，
          课案那段会直接崩，而不是给出「请先启动服务」的提示。

本文件把这三个坑都补上：参数外置 + 健康探测 + 失败降级。

```

> 上面那几行以 `#` 开头的（`# Request payload configuration` 等）是**课案原文里的注释行**。
> 注意它们不是 Markdown 标题，而是躺在代码围栏里的普通文本 —— 围栏内不做 Markdown 解析。
> 这类行在 percent 源码里要写「两个井号」，见文末「常见坑」第 1 条。

## 2. 完整版（一）：四平台对比与三段部署命令

这一节全部来自 `02_平台对比与部署_jxsd.py`。它的价值不在于「跑出什么」，
而在于**把课案里的五段命令逐行拆开讲**：

| 小节 | 课案出处 | 本机能不能真跑 |
|---|---|---|
| 2.3 四平台对比 | 课案「对比」那节 | 🟢 纯打印 |
| 2.4 n8n 部署 | 课案「n8n → 安装」 | ❌ 要 Linux 服务器 + Docker |
| 2.5 Dify 论文追踪 | 课案「dify → 论文追踪」 | ❌ 要在 Dify 界面里配 |
| 2.6 Langflow 安装/启动/生产 | 课案「langflow → 安装 / 启动 / 生产环境」 | ❌ 要 conda / Docker |
| 2.7 本机环境探测 | —— | 🟢 真探测本机 |

### 2.1 五段课案原文命令（原样保留成常量）

为什么把命令写成常量，而不是直接在 `print` 里写字符串？

1. 课案原文可以**逐字保留**，改动只发生在注释里，对照时不会失真；
2. 这些块会被后面的 `print_command_block()` 打印 + 配一段「逐行说明」，
   把「原文」和「讲解」分成两个来源，读者一眼能分清哪句是课案的、哪句是补充的；
3. 五段集中放在一起，方便横向比较各家的安装方式。

> ⚠️ 这五段都是**给别人在自己机器上执行**的原文，本节**不会真的执行它们** ——
> 本节只做打印 + 本机环境探测（2.7）。
>
> ★ 口令一律用占位符 ★ ——
> 课案 Langflow 生产环境那步写的是 `LANGFLOW_SUPERUSER_PASSWORD=123456`，
> 本文件改成 `LANGFLOW_SUPERUSER_PASSWORD=YOUR_LANGFLOW_SUPERUSER_PASSWORD`，
> 绝不把任何真实口令写进代码。

In [ ]:
# --- ① n8n：课案「n8n → 安装」里的 shell 块（Linux 服务器上执行）---
N8N_INSTALL_CMD = """mkdir -p /data/n8n
# 赋予node用户（UID=1000）读写权限
sudo chown -R 1000:1000 /data/n8n
sudo chmod -R 755 /data/n8n

docker run -d --rm --name n8n-tunnel -p 5678:5678 -v n8n_test_data:/home/node/.n8n -e N8N_SECURE_COOKIE=false n8nio/n8n:latest start --tunnel
"""

# --- ② Dify：课案「dify → 论文追踪」里配置的 HTTP 请求工具 URL 模板 ---
DIFY_ARXIV_URLS = """https://export.arxiv.org/api/query?search_query=all:{{topic}}&start=0&max_results=5&sortBy=submittedDate


https://export.arxiv.org/api/query?search_query=cat:cs.AI&max_results=5
"""

# --- ③ Langflow：课案「langflow → 安装 → 开发环境」的 conda 流程 ---
LANGFLOW_DEV_CMD = """# 创建 Conda 环境
conda create -n langflow python=3.11 -y

# 激活环境
conda activate langflow

# 安装高性能依赖安装器
python -m pip install --upgrade uv

# 使用 uv 安装 Langflow
uv pip install --python "$env:CONDA_PREFIX\\python.exe" langflow

# 验证安装
langflow --version
python -m pip check

# 设置当前 PowerShell 会话的环境变量
# 完全关闭 SSRF 防护。这样 Langflow 可以访问本机数据库、Ollama、Docker 服务等，
# 但如果 Langflow 暴露到公网，会有安全风险
$env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"
$env:LANGFLOW_SSRF_ALLOWED_HOSTS = "localhost,127.0.0.1"

# 启动 Langflow
langflow run --host 127.0.0.1 --port 7860
"""

# --- ④ Langflow：课案「langflow → 启动 → 生产环境」的 docker compose 流程 ---
LANGFLOW_PROD_CMD = """git clone https://github.com/langflow-ai/langflow.git
cd langflow/docker_example
# 创建一个带有 Langflow 管理员密码的文件：.env
# 默认管理员用户名是 langflow
LANGFLOW_SUPERUSER_PASSWORD=YOUR_LANGFLOW_SUPERUSER_PASSWORD
docker compose up
# 访问 http://localhost:7860/
"""

# --- ⑤ Langflow：课案「搭建 RAG 问答」里下载本地向量模型的命令 ---
OLLAMA_CMD = """ollama pull bge-m3
ollama run bge-m3  "这是一个测试文本"


ollama tun qwen3
"""

### 2.2 三个打印辅助函数

源文件里有三个小工具，后面每个小节都要用：

| 函数 | 干什么 | 为什么需要 |
|---|---|---|
| `display_width()` | 按「中文算 2 列」算显示宽度 | 不这么做，含中文的 ASCII 表会歪成一团 |
| `print_table()` | 打印一张自适应的 ASCII 表 | 课案的对比表在源文件里是代码，得打出来才看得见 |
| `print_command_block()` | 打印「课案原文 + 逐行中文说明」 | 把「原文」和「讲解」分开成两个来源 |

> 这一格**没有输出** —— 它只定义函数。

In [ ]:
def display_width(text: str) -> int:
    """中文按 2 列算，否则中文表格会歪。"""
    return sum(2 if "\u2e80" <= ch <= "\u9fff" or "\uff00" <= ch <= "\uff60" else 1 for ch in text)


def print_table(headers: list[str], rows: list[list[str]], widths: list[int]) -> None:
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(line)
    print("| " + " | ".join(h + " " * (w - display_width(h)) for h, w in zip(headers, widths)) + " |")
    print(line)
    for row in rows:
        cells = [c.split("\n") for c in row]
        for i in range(max(len(c) for c in cells)):
            parts = [
                (c[i] if i < len(c) else "") + " " * (w - display_width(c[i] if i < len(c) else ""))
                for c, w in zip(cells, widths)
            ]
            print("| " + " | ".join(parts) + " |")
        print(line)


def print_command_block(title: str, command: str, notes: list[str]) -> None:
    """打印一段课案原文命令，然后逐行给出中文注释。"""
    print(f"  【{title}】课案原文：")
    print("  " + "-" * 74)
    for line in command.rstrip("\n").splitlines():
        print("  | " + line)
    print("  " + "-" * 74)
    print("  逐行说明：")
    for note in notes:
        print("    · " + note)
    print()

### 2.3 四平台对比（课案「对比」那节）

这一格就是把 1.1 那张 Markdown 表**再打印成 ASCII 表**。
两处看似重复，其实用途不同：

- 1.1 的 Markdown 表是给你**读**的（可排序、可加列、能渲染）；
- 这里的 ASCII 表是源文件**真的会打印**的内容，保留它才能对照源码输出。

关键是 `widths=[12, 48, 8]` 这组宽度 —— 中文按 2 列算，所以 48 宽的那列
正好放得下「可视化 LangChain/LangGraph，」这半行，于是源文件里那个
`\n` 手动换行的两行单元格能对齐。

In [ ]:
def section_compare() -> None:
    print("=" * 78)
    print("1. 四平台对比（课案「对比」那节）")
    print("=" * 78)
    print()

    print_table(
        ["平台", "定位", "开源"],
        [
            ["Coze", "面向普通用户，适合日常使用", "否"],
            ["Dify", "面向开发者", "是"],
            ["n8n", "面向专业开发者", "是"],
            ["Langflow", "可视化 LangChain/LangGraph，\n面向专业开发者，适合定制化开发", "是"],
        ],
        [12, 48, 8],
    )
    print()
    print("  地址：Coze https://code.coze.cn/home    Dify https://cloud.dify.ai/")
    print()
    print(
        """  补充说明（课案表格之外的取舍依据）：

    · 开源与否决定的不是「免不免费」，而是「能不能私有化部署、能不能改源码」。
      Coze 闭源 → 数据必须过它的云；Dify / n8n / Langflow 都能整包丢进内网。
    · 「面向开发者」的差别在于抽象层次：
        Dify   把 RAG 链路（分段、检索、重排、引用）做成了开箱即用的节点；
        n8n    把「集成」做成了几百个现成节点，强项是跨系统自动化；
        Langflow 直接暴露 LangChain/LangGraph 的组件，能细调到代码级，
                 代价是你要懂 LangChain 的概念（Chain / Retriever / Tool）。
    · 本仓库 Agent/ 目录下那些 LangGraph 代码，配一个 Langflow 界面调试是最顺的；
      如果只是想把「每天抓 GitHub 热榜发到群里」跑起来，n8n 更省事。
"""
    )


section_compare()

### 预期输出

```text
==============================================================================
1. 四平台对比（课案「对比」那节）
==============================================================================

+--------------+--------------------------------------------------+----------+
| 平台         | 定位                                             | 开源     |
+--------------+--------------------------------------------------+----------+
| Coze         | 面向普通用户，适合日常使用                       | 否       |
+--------------+--------------------------------------------------+----------+
| Dify         | 面向开发者                                       | 是       |
+--------------+--------------------------------------------------+----------+
| n8n          | 面向专业开发者                                   | 是       |
+--------------+--------------------------------------------------+----------+
| Langflow     | 可视化 LangChain/LangGraph，                     | 是       |
|              | 面向专业开发者，适合定制化开发                   |          |
+--------------+--------------------------------------------------+----------+

  地址：Coze https://code.coze.cn/home    Dify https://cloud.dify.ai/

  补充说明（课案表格之外的取舍依据）：

    · 开源与否决定的不是「免不免费」，而是「能不能私有化部署、能不能改源码」。
      Coze 闭源 → 数据必须过它的云；Dify / n8n / Langflow 都能整包丢进内网。
    · 「面向开发者」的差别在于抽象层次：
        Dify   把 RAG 链路（分段、检索、重排、引用）做成了开箱即用的节点；
        n8n    把「集成」做成了几百个现成节点，强项是跨系统自动化；
        Langflow 直接暴露 LangChain/LangGraph 的组件，能细调到代码级，
                 代价是你要懂 LangChain 的概念（Chain / Retriever / Tool）。
    · 本仓库 Agent/ 目录下那些 LangGraph 代码，配一个 Langflow 界面调试是最顺的；
      如果只是想把「每天抓 GitHub 热榜发到群里」跑起来，n8n 更省事。
```

**看表得到的两个结论**：

- `Langflow` 那一行的单元格里有个 `\n`，被 `print_table()` 拆成**两行**打印，
  而且第二行左边留了 14 个空格（`| ` + 12 宽 + ` | `）—— 这就是 `display_width()`
  存在的理由：中文算 2 列，不用它对齐就是歪的；
- 四家里**只有 Coze 闭源**。对一个要上内网的项目来说，这一列基本就决定了答案。

### 2.4 n8n 部署（课案「n8n → 安装」）

课案那段 `docker run` 里有**一个很容易看漏的细节**：
前面刚 `mkdir -p /data/n8n` 并 `chown` 好，后面挂的却是**命名卷**
`n8n_test_data:/home/node/.n8n` —— 那个 `/data/n8n` 根本没被挂进去。
两者不冲突，但要清楚**真正的数据在命名卷里**，
`docker volume inspect n8n_test_data` 才能查到宿主机上的实际路径。

In [ ]:
def section_n8n() -> None:
    print("=" * 78)
    print("2. n8n：安装（课案「n8n → 安装」）")
    print("=" * 78)
    print()
    print_command_block(
        "n8n docker 部署（在 Linux 服务器上执行）",
        N8N_INSTALL_CMD,
        [
            "mkdir -p /data/n8n —— 建数据目录。n8n 的工作流定义、凭证都存这里，"
            "所以必须落在宿主机上，不能只靠容器内的匿名卷。",
            "# 赋予node用户（UID=1000）读写权限 —— n8n 官方镜像里的 node 用户 UID 固定是 1000，"
            "容器以该用户运行；不 chown 的话容器起得来但写不进数据，工作流保存会失败。",
            "sudo chown -R 1000:1000 /data/n8n —— 把属主改成 1000:1000。",
            "sudo chmod -R 755 /data/n8n —— 属主可读写执行、其他用户可读可执行。",
            "docker run —— 注意课案这里挂的是【命名卷】n8n_test_data:/home/node/.n8n，"
            "而不是上面刚建的 /data/n8n；两者不冲突，但要知道真正的数据在命名卷里"
            "（docker volume inspect n8n_test_data 可以查到宿主机路径）。",
            "-d --rm —— 后台运行，容器退出后自动删除（--rm 与持久化数据是兼容的，"
            "因为数据在卷里；但不适合生产，生产别加 --rm）。",
            "--name n8n-tunnel —— 容器名，后面 docker logs/stop 都用它。",
            "-p 5678:5678 —— 宿主机 5678 → 容器 5678，n8n 默认端口。",
            "-e N8N_SECURE_COOKIE=false —— 允许用 http（非 https）访问。"
            "n8n 默认要求 https 才给登录，本地调试必须关掉，否则登录页一直转圈。",
            "n8nio/n8n:latest start --tunnel —— 用官方镜像启动。"
            "start --tunnel 会开一条隧道，让外部服务（如 GitHub Webhook）能回调到你本机，"
            "开发 webhook 流程时必开；纯本地用可以去掉 --tunnel。",
            "国内网络拉取慢的话，可换镜像加速或指定具体版本号代替 :latest。",
        ],
    )
    print(
        """  课案「使用（github热榜）」那条流程的思路（图看不懂也能照做）：
      Schedule Trigger（定时）→ HTTP Request（GitHub Trending 页面）
      → HTML/Code 节点提取仓库名 → 格式化 → 推送到 IM。
      这一步的价值在于演示 n8n 的核心卖点：不用写代码就能把
      「定时 + HTTP + 解析 + 通知」串起来。
"""
    )


section_n8n()

### 预期输出

```text
==============================================================================
2. n8n：安装（课案「n8n → 安装」）
==============================================================================

  【n8n docker 部署（在 Linux 服务器上执行）】课案原文：
  --------------------------------------------------------------------------
  | mkdir -p /data/n8n
  | # 赋予node用户（UID=1000）读写权限
  | sudo chown -R 1000:1000 /data/n8n
  | sudo chmod -R 755 /data/n8n
  |
  | docker run -d --rm --name n8n-tunnel -p 5678:5678 -v n8n_test_data:/home/node/.n8n -e N8N_SECURE_COOKIE=false n8nio/n8n:latest start --tunnel
  --------------------------------------------------------------------------
  逐行说明：
    · mkdir -p /data/n8n —— 建数据目录。n8n 的工作流定义、凭证都存这里，所以必须落在宿主机上，不能只靠容器内的匿名卷。
    · # 赋予node用户（UID=1000）读写权限 —— n8n 官方镜像里的 node 用户 UID 固定是 1000，容器以该用户运行；不 chown 的话容器起得来但写不进数据，工作流保存会失败。
    · sudo chown -R 1000:1000 /data/n8n —— 把属主改成 1000:1000。
    · sudo chmod -R 755 /data/n8n —— 属主可读写执行、其他用户可读可执行。
    · docker run —— 注意课案这里挂的是【命名卷】n8n_test_data:/home/node/.n8n，而不是上面刚建的 /data/n8n；两者不冲突，但要知道真正的数据在命名卷里（docker volume inspect n8n_test_data 可以查到宿主机路径）。
    · -d --rm —— 后台运行，容器退出后自动删除（--rm 与持久化数据是兼容的，因为数据在卷里；但不适合生产，生产别加 --rm）。
    · --name n8n-tunnel —— 容器名，后面 docker logs/stop 都用它。
    · -p 5678:5678 —— 宿主机 5678 → 容器 5678，n8n 默认端口。
    · -e N8N_SECURE_COOKIE=false —— 允许用 http（非 https）访问。n8n 默认要求 https 才给登录，本地调试必须关掉，否则登录页一直转圈。
    · n8nio/n8n:latest start --tunnel —— 用官方镜像启动。start --tunnel 会开一条隧道，让外部服务（如 GitHub Webhook）能回调到你本机，开发 webhook 流程时必开；纯本地用可以去掉 --tunnel。
    · 国内网络拉取慢的话，可换镜像加速或指定具体版本号代替 :latest。

  课案「使用（github热榜）」那条流程的思路（图看不懂也能照做）：
      Schedule Trigger（定时）→ HTTP Request（GitHub Trending 页面）
      → HTML/Code 节点提取仓库名 → 格式化 → 推送到 IM。
      这一步的价值在于演示 n8n 的核心卖点：不用写代码就能把
      「定时 + HTTP + 解析 + 通知」串起来。
```

> **这一格在本机是「只打印、不执行」** —— 那段 `docker run` 是给 Linux 服务器用的，
> 本 notebook 一行都不会真的跑它。要真的体验 n8n，最省事的是用 Docker Desktop：
> `docker run -d --rm -p 5678:5678 -v n8n_test_data:/home/node/.n8n -e N8N_SECURE_COOKIE=false n8nio/n8n:latest`
> （去掉 `--tunnel` 和 `sudo chown`），然后浏览器开 <http://localhost:5678>。

### 2.5 Dify 论文追踪（课案「dify → 论文追踪」）

课案在 Dify 里配了一个 **arxiv 论文检索** 的 HTTP 请求工具。这一格把那个 URL 模板
逐参数拆开 —— 四个参数里 **`sortBy=submittedDate` 是最容易被漏掉的那个**：
不加它，arxiv 默认按相关度排序，做「论文追踪」拿到的就是一堆老论文。

In [ ]:
def section_dify() -> None:
    print("=" * 78)
    print("3. Dify：论文追踪（课案「dify → 论文追踪」）")
    print("=" * 78)
    print()
    print("  课案在 Dify 里配置的 arxiv 检索工具 URL 模板：")
    print("  " + "-" * 74)
    for line in DIFY_ARXIV_URLS.rstrip("\n").splitlines():
        print("  | " + line)
    print("  " + "-" * 74)
    print(
        """  逐参数说明（这是 Dify 里「自定义工具 / HTTP 请求节点」的典型配置）：

    · https://export.arxiv.org/api/query —— arxiv 官方 API，
      export 子域返回 Atom XML，比 www 页面更适合程序解析，且不限流那么狠。
    · search_query=all:{{topic}} —— 双花括号是【Dify 的模板变量】语法，
      运行时由用户在对话里填的 topic 替换。all: 表示在标题+摘要+作者里全文搜；
      换成 ti: 只搜标题、au: 只搜作者、cat: 只搜分类。
    · start=0&max_results=5 —— 从第 0 条开始取 5 条。分页就改 start。
    · sortBy=submittedDate —— 按投稿时间排序（默认按相关度）。
      做「论文追踪」必须加这个，否则拿到的是一堆老论文。
    · 第二条 https://export.arxiv.org/api/query?search_query=cat:cs.AI&max_results=5
      是「只看 cs.AI 分类的最新 5 篇」的固定 URL，适合做每日推送。

  接进 Dify 之后的一般流程：
      开始节点（收 topic）→ HTTP 请求节点（上面这个 URL）→
      LLM 节点（把 XML 结果总结成中文摘要）→ 结束节点（输出）。

  小坑：arxiv 返回的是 XML，Dify 的 HTTP 节点默认按 JSON 解析会失败，
       要么在节点里选「文本」响应类型再用代码节点解析，
       要么在 URL 后面加 &x=1 之类的干扰参数骗过缓存（课案没提，但社区常用）。
"""
    )


section_dify()

### 预期输出

```text
==============================================================================
3. Dify：论文追踪（课案「dify → 论文追踪」）
==============================================================================

  课案在 Dify 里配置的 arxiv 检索工具 URL 模板：
  --------------------------------------------------------------------------
  | https://export.arxiv.org/api/query?search_query=all:{{topic}}&start=0&max_results=5&sortBy=submittedDate
  |
  |
  | https://export.arxiv.org/api/query?search_query=cat:cs.AI&max_results=5
  --------------------------------------------------------------------------
  逐参数说明（这是 Dify 里「自定义工具 / HTTP 请求节点」的典型配置）：

    · https://export.arxiv.org/api/query —— arxiv 官方 API，
      export 子域返回 Atom XML，比 www 页面更适合程序解析，且不限流那么狠。
    · search_query=all:{{topic}} —— 双花括号是【Dify 的模板变量】语法，
      运行时由用户在对话里填的 topic 替换。all: 表示在标题+摘要+作者里全文搜；
      换成 ti: 只搜标题、au: 只搜作者、cat: 只搜分类。
    · start=0&max_results=5 —— 从第 0 条开始取 5 条。分页就改 start。
    · sortBy=submittedDate —— 按投稿时间排序（默认按相关度）。
      做「论文追踪」必须加这个，否则拿到的是一堆老论文。
    · 第二条 https://export.arxiv.org/api/query?search_query=cat:cs.AI&max_results=5
      是「只看 cs.AI 分类的最新 5 篇」的固定 URL，适合做每日推送。

  接进 Dify 之后的一般流程：
      开始节点（收 topic）→ HTTP 请求节点（上面这个 URL）→
      LLM 节点（把 XML 结果总结成中文摘要）→ 结束节点（输出）。

  小坑：arxiv 返回的是 XML，Dify 的 HTTP 节点默认按 JSON 解析会失败，
       要么在节点里选「文本」响应类型再用代码节点解析，
       要么在 URL 后面加 &x=1 之类的干扰参数骗过缓存（课案没提，但社区常用）。
```

> 注意输出里 `{{topic}}` 是**原样打印**的两个花括号 —— 它在打印时不会报 KeyError，
> 因为它躺在普通字符串里；只有当它是 f-string 时 `{{` 才会被转义成单个 `{`。
> 这一格是普通字符串，所以 Dify 的模板语法原样可见，正合用。

### 2.6 Langflow 安装与启动（课案「langflow → 安装 / 启动 / 生产环境」）

这一格最长，因为课案里 Langflow 给了**三套**东西：

| 段落 | 场景 | 关键点 |
|---|---|---|
| ① conda + uv 开发环境 | 自己电脑上跑起来 | `uv pip install --python "$env:CONDA_PREFIX\python.exe"` —— uv 不认 conda 环境，必须显式指 |
| ② docker compose 生产环境 | 部署到服务器 | 管理员口令 / `docker_example` 里有 PostgreSQL |
| ③ ollama 拉向量模型 | 搭 RAG 之前 | `bge-m3` 是向量模型，中文效果好且免费 |

**最容易踩的两个坑**（都在输出里点出来了）：

- `$env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"`：Langflow 默认**禁止工作流访问内网地址**，
  而你的本地数据库 / Ollama 都在内网，不关就一调一个报错。
  ⚠️ 只有在 Langflow **不暴露公网**时才关；更保险的做法是用
  `LANGFLOW_SSRF_ALLOWED_HOSTS` 只把 `localhost,127.0.0.1` 加进白名单。
- 课案原文 `ollama tun qwen3` 是**笔误**，应为 `ollama run qwen3`。
  本文件保留原文以便对照，没有替课案改字。

In [ ]:
def section_langflow() -> None:
    print("=" * 78)
    print("4. Langflow：安装与启动（课案「langflow → 安装 / 启动」）")
    print("=" * 78)
    print()
    print_command_block(
        "开发环境：conda 建环境 + uv 装包 + 启动",
        LANGFLOW_DEV_CMD,
        [
            "conda create -n langflow python=3.11 -y —— 单独开一个环境。"
            "Langflow 依赖树很重，和项目主环境混装极易冲突，这是必要的隔离。",
            "conda activate langflow —— 激活环境；后面所有命令都在这个环境里执行。",
            "python -m pip install --upgrade uv —— 装 uv 作为安装器。"
            "Langflow 依赖上百个包，用 uv 比 pip 快一个数量级。",
            'uv pip install --python "$env:CONDA_PREFIX\\python.exe" langflow —— '
            "关键点：uv 默认不认 conda 环境，必须用 --python 显式指向 "
            "$env:CONDA_PREFIX（conda 当前环境的根目录）下的解释器，"
            "否则会装到别处，装完 langflow 命令找不到。",
            "langflow --version / python -m pip check —— 验证安装 + 检查依赖冲突。",
            "$env:LANGFLOW_SSRF_PROTECTION_ENABLED = \"false\" —— 关掉 SSRF 防护。"
            "Langflow 默认禁止工作流访问内网地址，但你的本地数据库、Ollama 都在内网，"
            "不关就一调一个报错。⚠ 只有在 Langflow 不暴露公网时才关。",
            "$env:LANGFLOW_SSRF_ALLOWED_HOSTS = \"localhost,127.0.0.1\" —— "
            "更保险的做法：不整体关闭，只把本机加进白名单。",
            "langflow run --host 127.0.0.1 --port 7860 —— 启动。"
            "只监听 127.0.0.1 是安全的默认；要让同事访问才改成 0.0.0.0。",
            "启动后浏览器访问 http://localhost:7860/，就能拖节点搭流程了。",
        ],
    )
    print("  课案「启动」那节就是上面这段的浓缩版（不需要重装时只跑这三行）：")
    print("      conda activate langflow")
    print('      $env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"')
    print("      langflow run --host 127.0.0.1 --port 7860")
    print()

    print_command_block(
        "生产环境：docker compose（课案「生产环境」五步）",
        LANGFLOW_PROD_CMD,
        [
            "git clone https://github.com/langflow-ai/langflow.git —— 拉源码，"
            "生产部署用的是仓库里现成的 docker_example，不是 pip 包。",
            "cd langflow/docker_example —— 切到示例目录，里面有 docker-compose.yml"
            "（含 Langflow + PostgreSQL 两个服务，生产必须外接数据库，不能用 SQLite）。",
            "创建 .env 文件 —— compose 会自动读同目录的 .env。",
            "LANGFLOW_SUPERUSER_PASSWORD=... —— 管理员口令。"
            "★ 课案原文写的是 123456，本文件按规范改成占位符 "
            "YOUR_LANGFLOW_SUPERUSER_PASSWORD；真实部署请用强口令，"
            "并且 .env 绝对不能提交进 Git。",
            "默认管理员用户名是 langflow —— 用户名不用配，是固定的。",
            "docker compose up —— 起服务；加 -d 可以后台运行（课案没加）。",
            "访问 http://localhost:7860/ —— 生产环境同样是 7860 端口，"
            "和开发环境一致，所以本机不要同时开两个。",
        ],
    )
    print_command_block(
        "搭 RAG 问答之前：用 ollama 拉本地向量模型（课案「搭建 RAG 问答」第 2 步）",
        OLLAMA_CMD,
        [
            "ollama pull bge-m3 —— 拉 BGE-M3 向量模型（多语言，中文效果好），"
            "在 Langflow 里把它配成 Embedding 节点，就不用花 OpenAI 的钱。",
            'ollama run bge-m3 "这是一个测试文本" —— 验证模型能跑。',
            "ollama tun qwen3 —— 课案原文如此，应为 `ollama run qwen3`（笔误），"
            "作用是拉一个本地大模型作为 LLM 节点；本文件保留原文以便对照。",
            "前置：先按 https://ollama.com/ 装好 ollama 并让它常驻。",
        ],
    )


section_langflow()

### 预期输出

```text
==============================================================================
4. Langflow：安装与启动（课案「langflow → 安装 / 启动」）
==============================================================================

  【开发环境：conda 建环境 + uv 装包 + 启动】课案原文：
  --------------------------------------------------------------------------
  | # 创建 Conda 环境
  | conda create -n langflow python=3.11 -y
  |
  | # 激活环境
  | conda activate langflow
  |
  | # 安装高性能依赖安装器
  | python -m pip install --upgrade uv
  |
  | # 使用 uv 安装 Langflow
  | uv pip install --python "$env:CONDA_PREFIX\python.exe" langflow
  |
  | # 验证安装
  | langflow --version
  | python -m pip check
  |
  | # 设置当前 PowerShell 会话的环境变量
  | # 完全关闭 SSRF 防护。这样 Langflow 可以访问本机数据库、Ollama、Docker 服务等，
  | # 但如果 Langflow 暴露到公网，会有安全风险
  | $env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"
  | $env:LANGFLOW_SSRF_ALLOWED_HOSTS = "localhost,127.0.0.1"
  |
  | # 启动 Langflow
  | langflow run --host 127.0.0.1 --port 7860
  --------------------------------------------------------------------------
  逐行说明：
    · conda create -n langflow python=3.11 -y —— 单独开一个环境。Langflow 依赖树很重，和项目主环境混装极易冲突，这是必要的隔离。
    · conda activate langflow —— 激活环境；后面所有命令都在这个环境里执行。
    · python -m pip install --upgrade uv —— 装 uv 作为安装器。Langflow 依赖上百个包，用 uv 比 pip 快一个数量级。
    · uv pip install --python "$env:CONDA_PREFIX\python.exe" langflow —— 关键点：uv 默认不认 conda 环境，必须用 --python 显式指向 $env:CONDA_PREFIX（conda 当前环境的根目录）下的解释器，否则会装到别处，装完 langflow 命令找不到。
    · langflow --version / python -m pip check —— 验证安装 + 检查依赖冲突。
    · $env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false" —— 关掉 SSRF 防护。Langflow 默认禁止工作流访问内网地址，但你的本地数据库、Ollama 都在内网，不关就一调一个报错。⚠ 只有在 Langflow 不暴露公网时才关。
    · $env:LANGFLOW_SSRF_ALLOWED_HOSTS = "localhost,127.0.0.1" —— 更保险的做法：不整体关闭，只把本机加进白名单。
    · langflow run --host 127.0.0.1 --port 7860 —— 启动。只监听 127.0.0.1 是安全的默认；要让同事访问才改成 0.0.0.0。
    · 启动后浏览器访问 http://localhost:7860/，就能拖节点搭流程了。

  课案「启动」那节就是上面这段的浓缩版（不需要重装时只跑这三行）：
      conda activate langflow
      $env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"
      langflow run --host 127.0.0.1 --port 7860

  【生产环境：docker compose（课案「生产环境」五步）】课案原文：
  --------------------------------------------------------------------------
  | git clone https://github.com/langflow-ai/langflow.git
  | cd langflow/docker_example
  | # 创建一个带有 Langflow 管理员密码的文件：.env
  | # 默认管理员用户名是 langflow
  | LANGFLOW_SUPERUSER_PASSWORD=YOUR_LANGFLOW_SUPERUSER_PASSWORD
  | docker compose up
  | # 访问 http://localhost:7860/
  --------------------------------------------------------------------------
  逐行说明：
    · git clone https://github.com/langflow-ai/langflow.git —— 拉源码，生产部署用的是仓库里现成的 docker_example，不是 pip 包。
    · cd langflow/docker_example —— 切到示例目录，里面有 docker-compose.yml（含 Langflow + PostgreSQL 两个服务，生产必须外接数据库，不能用 SQLite）。
    · 创建 .env 文件 —— compose 会自动读同目录的 .env。
    · LANGFLOW_SUPERUSER_PASSWORD=... —— 管理员口令。★ 课案原文写的是 123456，本文件按规范改成占位符 YOUR_LANGFLOW_SUPERUSER_PASSWORD；真实部署请用强口令，并且 .env 绝对不能提交进 Git。
    · 默认管理员用户名是 langflow —— 用户名不用配，是固定的。
    · docker compose up —— 起服务；加 -d 可以后台运行（课案没加）。
    · 访问 http://localhost:7860/ —— 生产环境同样是 7860 端口，和开发环境一致，所以本机不要同时开两个。

  【搭 RAG 问答之前：用 ollama 拉本地向量模型（课案「搭建 RAG 问答」第 2 步）】课案原文：
  --------------------------------------------------------------------------
  | ollama pull bge-m3
  | ollama run bge-m3  "这是一个测试文本"
  |
  |
  | ollama tun qwen3
  --------------------------------------------------------------------------
  逐行说明：
    · ollama pull bge-m3 —— 拉 BGE-M3 向量模型（多语言，中文效果好），在 Langflow 里把它配成 Embedding 节点，就不用花 OpenAI 的钱。
    · ollama run bge-m3 "这是一个测试文本" —— 验证模型能跑。
    · ollama tun qwen3 —— 课案原文如此，应为 `ollama run qwen3`（笔误），作用是拉一个本地大模型作为 LLM 节点；本文件保留原文以便对照。
    · 前置：先按 https://ollama.com/ 装好 ollama 并让它常驻。
```

**三套命令的共同结论**：Langflow 只在开发环境「轻」，生产环境一样是
Docker + 外接 PostgreSQL + 强口令那一套。**别把 conda 那套当生产方案。**

### 2.7 本机环境探测：上面有三段命令依赖 Docker

前面五段命令里有**两段真的要用 Docker**（n8n 的 `docker run`、Langflow 的 `docker compose`）。
这一格做本机实测，而且分两层：

| 检查 | 为什么不能只看第一层 |
|---|---|
| `shutil.which("docker")` | 只说明 **命令在**，不代表守护进程在跑 |
| `docker info --format {{.ServerVersion}}` | 命令在但守护进程没起来（Docker Desktop 没启动）时，这里才是真相 |

顺带还查两件事：`langflow` 命令装没装、`7860` 端口有没有人在听
—— 后者和 3.4 节的健康探测是同一件事的两种写法（一个用 socket，一个用 HTTP）。

> 源文件把 `import shutil` / `import subprocess` 写在文件头；本 notebook 里
> `shutil` 在「0.1 前置条件自检」那格就导入了，`subprocess` 在这一格补上。

In [ ]:
import subprocess


def section_check_docker() -> None:
    print("=" * 78)
    print("5. 本机环境检查：上面有三段命令依赖 Docker")
    print("=" * 78)

    docker_path = shutil.which("docker")
    if docker_path is None:
        print("  ✗ 本机没有找到 docker 命令。")
        print("    → n8n 那段 docker run、Langflow 生产环境的 docker compose up 都跑不了。")
        print("    → 装 Docker Desktop for Windows：https://www.docker.com/products/docker-desktop/")
        print("    → 装完重开终端，再跑本文件确认。")
        print("    → 只想用 Langflow 的话，走第 4 节的 conda 开发环境方式，不需要 Docker。")
        return

    print(f"  ✓ 找到 docker：{docker_path}")

    # `docker info` 会去连守护进程：命令存在 ≠ 守护进程在跑（Docker Desktop 没启动就是这种）
    try:
        proc = subprocess.run(
            ["docker", "info", "--format", "{{.ServerVersion}}"],
            capture_output=True,
            text=True,
            timeout=20,
            check=False,
        )
    except (subprocess.TimeoutExpired, OSError) as exc:
        print(f"  ！执行 docker info 失败：{type(exc).__name__}: {exc}")
        return

    if proc.returncode == 0 and proc.stdout.strip():
        print(f"  ✓ Docker 守护进程在跑，Server 版本：{proc.stdout.strip()}")
        print("    → 第 2 节的 n8n 命令、第 4 节的 docker compose 命令都可以直接用。")
        print("    → n8n 起来后访问 http://localhost:5678 设置管理员账号。")
    else:
        print("  ！docker 命令在，但守护进程没响应（Docker Desktop 没启动？）")
        detail = (proc.stderr or proc.stdout).strip().splitlines()
        if detail:
            print(f"    最后一行输出：{detail[-1][:160]}")
        print("    → 启动 Docker Desktop，等托盘图标变绿后重跑本文件。")

    # 顺带看看 langflow 是否已装、7860 是否已被占用（和 01 那节的探测呼应）
    print()
    print("  顺带检查 Langflow 相关的两个前提：")
    langflow_path = shutil.which("langflow")
    print(f"    langflow 命令：{langflow_path if langflow_path else '未安装（走第 4 节的 conda 安装流程）'}")

    import socket

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1.0)
        listening = sock.connect_ex(("127.0.0.1", 7860)) == 0
    print(f"    127.0.0.1:7860 ：{'已在监听（Langflow 可能正在运行）' if listening else '没有服务监听（Langflow 未启动）'}")
    if not listening:
        print("    → 此时跑 01_langflow_api_jxsd.py 会走 dry-run 分支并打印中文排障提示，属预期行为。")


section_check_docker()

### 预期输出

本机（Docker Desktop 已在运行）实测输出：

```text
==============================================================================
5. 本机环境检查：上面有三段命令依赖 Docker
==============================================================================
  ✓ 找到 docker：C:\Program Files\Docker\Docker\resources\bin\docker.EXE
  ✓ Docker 守护进程在跑，Server 版本：29.7.2
    → 第 2 节的 n8n 命令、第 4 节的 docker compose 命令都可以直接用。
    → n8n 起来后访问 http://localhost:5678 设置管理员账号。

  顺带检查 Langflow 相关的两个前提：
    langflow 命令：未安装（走第 4 节的 conda 安装流程）
    127.0.0.1:7860 ：没有服务监听（Langflow 未启动）
    → 此时跑 01_langflow_api_jxsd.py 会走 dry-run 分支并打印中文排障提示，属预期行为。
```

**三行都可能变，不必对号入座**：

| 行 | 什么情况下会变 |
|---|---|
| `找到 docker：<路径>` | 没装 Docker 时变成「✗ 本机没有找到 docker 命令」+ 安装指引 |
| `Server 版本` | 版本号随 Docker Desktop 升级而变；守护进程没启动时变成「守护进程没响应」 |
| `7860` 那一行 | 真起了 Langflow 就变成「已在监听（Langflow 可能正在运行）」 |

> 顺带说明：`subprocess.run(..., timeout=20)` 里的 20 秒不会真的等到 ——
> 守护进程正常时 `docker info` 是毫秒级返回的（本机实测 0.3 秒）。

## 3. 完整版（二）：把可视化流程当接口调

到这里平台已经选好、也装好了。最后一节回到代码：**流程搭完之后怎么用**。

这一节全部来自 `01_langflow_api_jxsd.py`。它要讲的就一件事：
那个 `/api/v1/run/<flow_id>` 接口**到底要传什么**。

> ⚠️ 这一节在本机是 **🔴 需外部服务的降级态**：本机没有启动 Langflow
> （2.7 节刚确认 7860 没人听），所以 3.3~3.7 全部走
> **健康探测失败 → 打印请求预览 → 打印中文排障指引** 这条路，不会真的发 POST。
> 这不是代码错误，是源文件里就设计好的降级分支。
> 真起了服务之后，同一个 `main()` 会自动走 3.8 节描述的真实调用分支。

### 3.1 payload 四字段：课案图解里那四个输入框

| 字段 | 含义 | 常见取值 |
|---|---|---|
| `output_type` | 你要什么形态的返回 | `chat`（对话）/ `text` / `json` / `dataframe` |
| `input_type` | 你给的是什么形态的输入 | `chat`（对话）/ `text` |
| `input_value` | 真正的输入内容 | 用户那句话 |
| `session_id` | 会话标识，决定「记不记得上文」 | `str(uuid.uuid4())`，一个会话一个 |

**`session_id` 是四个里唯一需要想一下的**：同一个 `session_id` 的多次调用会被
Langflow 当成同一轮会话（带上文记忆）。想连续对话就把它固定下来复用 ——
工程上一般由业务层的会话 ID 决定，而不是每次随机。下面的 `build_payload()`
每调一次就新生成一个 `uuid4()`，那是**单轮问答**的用法。

### 3.2 参数解析：命令行 > 环境变量 > 默认值

课案那三个坑（硬编码 api_key / flow_id、连接失败就崩）里，前两个靠**参数外置**解决。
优先级顺序在 `parse_args()` 里体现得很直白 ——
`argparse` 的 `default=` 直接读 `os.getenv(...)`，于是
「**没写 `--api-key` 就用环境变量**」是自动成立的：

| 参数 | 命令行 | 环境变量 | 兜底 |
|---|---|---|---|
| `--base-url` | `--base-url http://host:7860` | `LANGFLOW_BASE_URL` | `http://localhost:7860` |
| `--flow-id` | `--flow-id <ID>` | `LANGFLOW_FLOW_ID` | 占位符 `YOUR_FLOW_ID` |
| `--api-key` | `--api-key <KEY>` | `LANGFLOW_API_KEY` | 占位符 `YOUR_LANGFLOW_API_KEY` |
| `--input` | `--input "帮我总结这篇论文"` | —— | `橘醒时代`（与课案一致） |
| `--output-type` | `--output-type text` | —— | `chat` |
| `--timeout` | `--timeout 30` | —— | `60.0` 秒 |
| `--dry-run` | `--dry-run` | —— | 关（默认真发请求） |

这一格只定义函数，**没有输出**。

In [ ]:
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="调用 Langflow 的 /api/v1/run/<flow_id> 接口（api_key 与 flow_id 不硬编码）",
    )
    parser.add_argument(
        "--base-url",
        default=os.getenv("LANGFLOW_BASE_URL", "http://localhost:7860"),
        help="Langflow 服务地址，默认取环境变量 LANGFLOW_BASE_URL，再默认 http://localhost:7860",
    )
    parser.add_argument(
        "--flow-id",
        default=os.getenv("LANGFLOW_FLOW_ID", ""),
        help=f"流程 ID，默认取环境变量 LANGFLOW_FLOW_ID；为空则用占位符 {PLACEHOLDER_FLOW_ID}",
    )
    parser.add_argument(
        "--api-key",
        default=os.getenv("LANGFLOW_API_KEY", ""),
        help=f"访问密钥，默认取环境变量 LANGFLOW_API_KEY；为空则用占位符 {PLACEHOLDER_API_KEY}",
    )
    parser.add_argument(
        "--input",
        default="橘醒时代",  # 与课案保持一致的示例输入
        help="input_value，即真正发给流程的内容",
    )
    parser.add_argument(
        "--output-type",
        default="chat",
        choices=["chat", "text", "json", "dataframe"],
        help="output_type，想要什么形态的返回",
    )
    parser.add_argument(
        "--timeout",
        type=float,
        default=60.0,
        help="HTTP 超时秒数",
    )
    parser.add_argument(
        "--dry-run",
        action="store_true",
        help="只打印将要发送的请求，不发出去",
    )
    return parser.parse_args()

#### notebook 里的 `argparse` 陷阱（这一格是本 notebook 特有的）

源文件是**命令行脚本**，`parser.parse_args()` 读的是 `sys.argv`。
而 Jupyter 内核的 `sys.argv` 里装的是**内核自己的启动参数**
（形如 `['.../ipykernel_launcher.py', '-f', '.../kernel-xxxx.json']`）——
直接调 `parse_args()`，argparse 会把它当成「无法识别的参数」而 `SystemExit`。

解法：临时把 `sys.argv` 换成「脚本名 + 想模拟的命令行」，**用完立刻还原**。
下面这个 `run_main()` 把这件事包起来，第 3.6 节直接复用它。

In [ ]:
def run_main(argv: list[str] | None = None) -> int:
    """在 notebook 里模拟命令行执行，等价于源文件末尾的：

        if __name__ == "__main__":
            section_course()
            sys.exit(main())

    notebook 里没有 `__main__` 这一层，也不能用 `sys.exit()`（`SystemExit` 会把内核杀掉），
    所以改成「临时替换 sys.argv → 调用 main() → 无论成败都还原」。
    换掉 sys.argv 是必须的：内核自己的 sys.argv 里是 ipykernel 的启动参数
    （`-f .../kernel-xxxx.json`），argparse 见到会直接 SystemExit。
    """
    backup = sys.argv[:]
    sys.argv = ["01_langflow_api_jxsd.py", *(argv or [])]
    try:
        return main()
    finally:
        sys.argv = backup


_argv_backup = sys.argv[:]
sys.argv = ["01_langflow_api_jxsd.py"]
args_default = parse_args()
sys.argv = _argv_backup

print("① 不带任何参数（等价于 python 01_langflow_api_jxsd.py）—— 全部走默认值：")
print(f"   base_url   = {args_default.base_url}")
print(f"   flow_id    = {args_default.flow_id!r}          ← 环境变量没设 → 空串，后面会被换成占位符")
print(f"   api_key    = {args_default.api_key!r}          ← 同上")
print(f"   input      = {args_default.input!r}")
print(f"   output_type= {args_default.output_type}")
print(f"   timeout    = {args_default.timeout}")
print(f"   dry_run    = {args_default.dry_run}")
print()

_argv_backup = sys.argv[:]
sys.argv = ["01_langflow_api_jxsd.py", "--input", "帮我总结这篇论文", "--output-type", "text", "--dry-run"]
args_cli = parse_args()
sys.argv = _argv_backup

print("② 带命令行参数（--input / --output-type / --dry-run）—— 命令行覆盖默认值：")
print(f"   input      = {args_cli.input!r}")
print(f"   output_type= {args_cli.output_type}")
print(f"   dry_run    = {args_cli.dry_run!r}")

### 预期输出

```text
① 不带任何参数（等价于 python 01_langflow_api_jxsd.py）—— 全部走默认值：
   base_url   = http://localhost:7860
   flow_id    = ''          ← 环境变量没设 → 空串，后面会被换成占位符
   api_key    = ''          ← 同上
   input      = '橘醒时代'
   output_type= chat
   timeout    = 60.0
   dry_run    = False

② 带命令行参数（--input / --output-type / --dry-run）—— 命令行覆盖默认值：
   input      = '帮我总结这篇论文'
   output_type= text
   dry_run    = True
```

两处细节：

- `flow_id` / `api_key` 是 `''` 而不是占位符 —— 占位符的替换发生在 `main()` 里
  （`flow_id = args.flow_id or PLACEHOLDER_FLOW_ID`），`parse_args()` 只管从命令行/环境变量取。
  这样设计是为了让「**没配**」和「**配了但是空**」在这里长得一样，都退到占位符，
  而 dry-run 打印出来的请求一眼就能看出哪里还没配；
- 字符串用 `!r` 打印，所以 `input = '橘醒时代'` 带引号；布尔值和浮点数直接格式化，
  所以 `dry_run = False`、`timeout = 60.0` 不带引号。这不是风格问题：
  `!r` 能让**空串**（`''`）和**首尾空格**在终端里一目了然，不然输出里只剩一片空白。

### 3.3 `build_payload()`：四个字段怎么组装

这一格就是课案那段 payload 的照抄，但加了两处说明：

- `input_type` 固定成 `"chat"`：课案没有暴露这个参数（因为对话场景它总是 `chat`）；
- `session_id` 每次调用**新生成一个 uuid4**。这是「单轮问答」的用法，
  想连续对话就得把这个值固定下来复用。

In [ ]:
def build_payload(input_value: str, output_type: str) -> dict:
    """构造课案那段 payload —— 注意这里就是课案四个字段的照抄。"""
    payload = {
        "output_type": output_type,   # 要什么形态的返回
        "input_type": "chat",         # 给的是对话形态的输入
        "input_value": input_value,   # 真正的输入内容
    }
    # session_id 每次调用都新生成一个 uuid4：
    # 同一个 session_id 的多次调用会被 Langflow 当成同一轮会话（带上文记忆），
    # 想连续对话就把这个值固定下来复用 —— 工程上一般由业务层的会话 ID 决定。
    payload["session_id"] = str(uuid.uuid4())
    return payload


_p1 = build_payload("橘醒时代", "chat")
_p2 = build_payload("橘醒时代", "chat")

print("payload 长这样（中文不转义）：")
print(json.dumps(_p1, ensure_ascii=False, indent=2))
print()
print("同样输入、连着调两次，session_id 不同 —— 说明默认是「两条独立会话」：")
print("  第一次：", _p1["session_id"])
print("  第二次：", _p2["session_id"])
print("  相同？  ", _p1["session_id"] == _p2["session_id"])

### 预期输出

```text
payload 长这样（中文不转义）：
{
  "output_type": "chat",
  "input_type": "chat",
  "input_value": "橘醒时代",
  "session_id": "a7322ffc-5225-4c09-b120-7e41034e2f92"
}

同样输入、连着调两次，session_id 不同 —— 说明默认是「两条独立会话」：
  第一次： a7322ffc-5225-4c09-b120-7e41034e2f92
  第二次： 0e20e4e0-2cd4-4c52-993e-ce68787e7d09
  相同？   False
```

> ⚠️ **上面那两串 UUID 是本机实测那一轮的取值，每次跑都不一样**（`uuid.uuid4()` 是随机的），
> 要对照的是**形状**（8-4-4-4-12 的十六进制）和最后那个 `False`，不是具体字符串。

### 3.4 `probe_health()`：先问 `/health`，再决定发不发请求

这是补第三个坑（连接失败就崩）的关键。`/health` 是 Langflow 自带的健康检查端点，
**未鉴权、开销极小**，适合做前置探测。

返回值是 `(是否可达, 说明文字)`。三个 `except` 的**顺序**在这里是有讲究的：

```text
ConnectionError  ← 服务没起（TCP 连接直接被拒）
Timeout          ← 服务起了但卡住（比如正在初始化数据库）—— 要和「没起来」分开报
RequestException ← 上面两个的【父类】，必须放最后兜底
```

> 把 `RequestException` 写在前面，`ConnectionError` / `Timeout` 就**永远轮不到**，
> 报错会变成笼统的「连接出错」—— 这是 `requests` 里最常见的 except 顺序坑。

In [ ]:
def probe_health(base_url: str, timeout: float = 5.0) -> tuple[bool, str]:
    """探测 Langflow 是否活着。

    返回 (是否可达, 说明文字)。
    /health 是 Langflow 自带的健康检查端点，未鉴权、开销极小，适合做前置探测。
    """
    url = f"{base_url.rstrip('/')}/health"
    try:
        response = requests.get(url, timeout=timeout)
        # 能拿到响应就算「服务活着」，哪怕状态码不是 200 ——
        # 有了响应说明进程在监听，后面真实的 POST 失败会给出更具体的错误码。
        return True, f"GET {url} → HTTP {response.status_code}"
    except requests.exceptions.ConnectionError:
        # 最常见的失败：Langflow 没启动，TCP 连接直接被拒
        return False, f"GET {url} → 连接被拒绝（服务没起来）"
    except requests.exceptions.Timeout:
        # 服务在，但卡住了（比如正在初始化数据库）—— 和「没起来」要分开报
        return False, f"GET {url} → 超时（{timeout} 秒内没有响应）"
    except requests.exceptions.RequestException as exc:
        # RequestException 是上面两个的【父类】，必须放在最后兜底：
        # 一旦写在前面，ConnectionError / Timeout 就永远轮不到，报错会变模糊。
        return False, f"GET {url} → {type(exc).__name__}: {exc}"


alive, detail = probe_health("http://localhost:7860")
print("探测本机 7860：")
print("   是否可达：", alive)
print("   说明    ：", detail)
print()
print("另一个常见错误：base_url 末尾带斜杠会拼出 //health、//api/v1/run ——")
print("所以 probe_health() 内部先 rstrip('/')，main() 里也先 rstrip('/')：")
alive2, detail2 = probe_health("http://localhost:7860/")
print("   传 'http://localhost:7860/'  →", detail2)

### 预期输出

```text
探测本机 7860：
   是否可达： False
   说明    ： GET http://localhost:7860/health → 连接被拒绝（服务没起来）

另一个常见错误：base_url 末尾带斜杠会拼出 //health、//api/v1/run ——
所以 probe_health() 内部先 rstrip('/')，main() 里也先 rstrip('/')：
   传 'http://localhost:7860/'  → GET http://localhost:7860/health → 连接被拒绝（服务没起来）
```

**注意最后两行的 URL 是同一个** `http://localhost:7860/health` ——
这就是 `rstrip("/")` 的效果。不写它，第二个 URL 会变成
`http://localhost:7860//health`（大多数服务器对 `//` 是宽容的，但 `//api/v1/run` 就不一定了）。

真启动了 Langflow 时，第一段的「说明」会变成 `GET http://localhost:7860/health → HTTP 200`，
「是否可达」变成 `True` —— 那时 `main()` 就会走真实调用分支（见 3.8 节）。

### 3.5 `print_request_preview()`：dry-run 与真实调用共用一个打印函数

这一格只定义函数、**没有输出** —— 它的输出在下一节 `main()` 里。

把它单独拆出来有个工程上的理由：**dry-run 和真实调用走同一个预览函数**，
于是「你到底会发出去什么」和「你实际发出去什么」永远一致，
不会出现「预览看着对、真发的却是另一份 payload」这种事故。

`json.dumps(..., ensure_ascii=False)` 里的 `ensure_ascii=False` 是给中文用的：
默认的 `ensure_ascii=True` 会把「橘醒时代」转成 `\u6a58\u9192...`，
肉眼核对中文请求体时非常难受。

In [ ]:
def print_request_preview(method: str, url: str, headers: dict, payload: dict) -> None:
    print("  ── 请求预览 ──")
    print(f"    {method} {url}")
    for key, value in headers.items():
        print(f"    Header: {key}: {value}")
    print("    Body（JSON，中文不转义，方便肉眼核对）：")
    body = json.dumps(payload, ensure_ascii=False, indent=6)
    for line in body.splitlines():
        print("      " + line)

### 3.6 主流程 `main()`：本机走降级路径

`main()` 就是源文件的原样。它把前面几格串起来，并且**只有两条出口**：

```mermaid
graph TD
    A["parse_args()<br/>参数外置"] --> B["probe_health(base_url)"]
    B --> C{"dry_run<br/>或<br/>服务不可达？"}
    C -->|"是"| D["打印请求预览<br/>+ 中文排障指引<br/>return 0"]
    C -->|"否"| E["POST /api/v1/run/flow_id"]
    E --> F["HTTPError → 按 401/403/404 给不同提示<br/>return 1"]
    E --> G["其它 RequestException → return 1"]
    E --> H["✅ 打印原始响应体<br/>+ 从嵌套 JSON 里取回答文本<br/>return 0"]
```

本机没有 Langflow，所以走的是 `D` 那条。下面这一格会打印：

1. 参数来源与本次调用的目标；
2. 健康探测结果（连接被拒绝）；
3. 请求预览（**完整的 POST URL / Header / Body**，含随机 session_id）；
4. 一段中文排障指引（怎么装、怎么起、怎么拿 flow_id 和 api_key）。

In [ ]:
def main() -> int:
    # 参数优先级顺序就写在这里：命令行 > 环境变量 > 硬编码默认值。
    # argparse 的 default 直接读 os.getenv，所以「没写 --api-key 就用环境变量」是自动的。
    args = parse_args()
    base_url = args.base_url.rstrip("/")   # 去掉末尾斜杠：否则拼出 //health、//api/v1/run
    # 空值一律退回占位符：这样 dry-run 打印出来的请求长什么样，一眼就能看出哪里还没配
    flow_id = args.flow_id or PLACEHOLDER_FLOW_ID
    api_key = args.api_key or PLACEHOLDER_API_KEY

    print("=" * 78)
    print("2. 参数来源与本次调用的目标")
    print("=" * 78)
    print(f"    base_url    : {base_url}      （--base-url / LANGFLOW_BASE_URL）")
    print(f"    flow_id     : {flow_id}      （--flow-id / LANGFLOW_FLOW_ID）")
    # 密钥一律只显示前后几位，避免整串口令出现在终端/日志里
    shown_key = api_key if api_key.startswith("YOUR_") else f"{api_key[:4]}...{api_key[-4:]}"
    print(f"    api_key     : {shown_key}      （--api-key / LANGFLOW_API_KEY，只显示首尾）")
    print(f"    input_value : {args.input}")
    print(f"    output_type : {args.output_type}")
    print(f"    dry_run     : {args.dry_run}")

    url = f"{base_url}/api/v1/run/{flow_id}"
    payload = build_payload(args.input, args.output_type)
    headers = {"x-api-key": api_key, "Content-Type": "application/json"}

    print()
    print("=" * 78)
    print("3. 健康探测：先确认服务在不在")
    print("=" * 78)
    alive, detail = probe_health(base_url)
    print(f"    {detail}")
    print(f"    → {'服务可达，继续真实调用' if alive else '服务不可达，降级为 dry-run'}")

    # ---------- 分支 A：不可达 / 用户显式 dry-run ----------
    if args.dry_run or not alive:
        print()
        print("=" * 78)
        print("4. DRY-RUN：只打印将要发送的请求，不真的发出去")
        print("=" * 78)
        print_request_preview("POST", url, headers, payload)
        print()
        if not alive and not args.dry_run:
            print("=" * 78)
            print("5. 中文排障提示：Langflow 还没起来")
            print("=" * 78)
            print(
                f"""
    探测结果：{detail}

    这不是代码错误 —— 是 Langflow 服务没启动。按下面顺序处理：

    1) 装（未装过才需要，conda 环境方式）：
           conda create -n langflow python=3.11 -y
           conda activate langflow
           python -m pip install --upgrade uv
           uv pip install --python "$env:CONDA_PREFIX\\python.exe" langflow
           langflow --version

    2) 起（每次要用之前）：
           conda activate langflow
           $env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"
           $env:LANGFLOW_SSRF_ALLOWED_HOSTS = "localhost,127.0.0.1"
           langflow run --host 127.0.0.1 --port 7860

    3) 在浏览器打开 {base_url} ，拖一个流程出来，然后：
           · 流程页 URL 里的 /flow/<这一串> 就是 flow_id
           · 右上角 Settings → API Keys 生成密钥，填进环境变量：
                 $env:LANGFLOW_FLOW_ID = "<你的 flow_id>"
                 $env:LANGFLOW_API_KEY = "{PLACEHOLDER_API_KEY}"
           或者直接用命令行参数：--flow-id <ID> --api-key <KEY>

    4) 服务起来后重跑本文件，就会走真实调用分支。

    详细部署命令（含生产环境 docker compose 方式）见同目录的
    02_平台对比与部署_jxsd.py。
"""
            )
        else:
            print("    （--dry-run 指定，跳过真实请求。要去掉这个参数并确保 Langflow 已启动。）")
        return 0

    # ---------- 分支 B：服务可达，真的发请求 ----------
    print()
    print("=" * 78)
    print("4. 真实调用：POST " + url)
    print("=" * 78)
    print_request_preview("POST", url, headers, payload)
    print()

    try:
        response = requests.request("POST", url, json=payload, headers=headers, timeout=args.timeout)
        response.raise_for_status()  # 4xx/5xx 直接抛，交给下面统一处理
    except requests.exceptions.HTTPError as exc:
        print(f"  ！HTTP 错误：{exc}")
        print(f"    HTTP {response.status_code}，响应体：")
        print("    " + response.text[:800])
        if response.status_code in (401, 403):
            print(f"    → 密钥不对或没传。检查 --api-key / LANGFLOW_API_KEY（占位符是 {PLACEHOLDER_API_KEY}）。")
        elif response.status_code == 404:
            print(f"    → flow_id 不存在。检查 --flow-id / LANGFLOW_FLOW_ID（占位符是 {PLACEHOLDER_FLOW_ID}）。")
        return 1
    except requests.exceptions.RequestException as exc:
        print(f"  ！请求失败：{type(exc).__name__}: {exc}")
        return 1

    print(f"  ✅ HTTP {response.status_code}")
    print("  ── 原始响应体（课案就是 print(response.text)）──")
    print("  " + "-" * 74)
    for line in response.text[:2000].splitlines():
        print("  | " + line)
    print("  " + "-" * 74)

    # Langflow 的返回是嵌套 JSON，真正的话术藏在 outputs[0].outputs[0].outputs[0].results.message.text
    # 这里顺手把它捞出来，免得每次都要肉眼在 JSON 里翻。
    print("  ── 试着直接取出回答文本 ──")
    try:
        data = response.json()
        text = (
            data.get("outputs", [{}])[0]
            .get("outputs", [{}])[0]
            .get("outputs", [{}])[0]
            .get("results", {})
            .get("message", {})
            .get("text")
        )
        print("  " + (text if text else "（这个流程的返回结构不是标准 chat 输出，请看上面的原始响应体）"))
    except ValueError:
        print("  （响应不是 JSON，跳过解析）")
    return 0


run_main([])

### 预期输出

本机（7860 无服务）实测输出 —— **注意第 4 段那个完整的请求预览，那就是 dry-run 的全部价值**：

```text
==============================================================================
2. 参数来源与本次调用的目标
==============================================================================
    base_url    : http://localhost:7860      （--base-url / LANGFLOW_BASE_URL）
    flow_id     : YOUR_FLOW_ID      （--flow-id / LANGFLOW_FLOW_ID）
    api_key     : YOUR_LANGFLOW_API_KEY      （--api-key / LANGFLOW_API_KEY，只显示首尾）
    input_value : 橘醒时代
    output_type : chat
    dry_run     : False

==============================================================================
3. 健康探测：先确认服务在不在
==============================================================================
    GET http://localhost:7860/health → 连接被拒绝（服务没起来）
    → 服务不可达，降级为 dry-run

==============================================================================
4. DRY-RUN：只打印将要发送的请求，不真的发出去
==============================================================================
  ── 请求预览 ──
    POST http://localhost:7860/api/v1/run/YOUR_FLOW_ID
    Header: x-api-key: YOUR_LANGFLOW_API_KEY
    Header: Content-Type: application/json
    Body（JSON，中文不转义，方便肉眼核对）：
      {
            "output_type": "chat",
            "input_type": "chat",
            "input_value": "橘醒时代",
            "session_id": "4b2e2761-b281-4009-a739-830f96d85028"
      }

==============================================================================
5. 中文排障提示：Langflow 还没起来
==============================================================================

    探测结果：GET http://localhost:7860/health → 连接被拒绝（服务没起来）

    这不是代码错误 —— 是 Langflow 服务没启动。按下面顺序处理：

    1) 装（未装过才需要，conda 环境方式）：
           conda create -n langflow python=3.11 -y
           conda activate langflow
           python -m pip install --upgrade uv
           uv pip install --python "$env:CONDA_PREFIX\python.exe" langflow
           langflow --version

    2) 起（每次要用之前）：
           conda activate langflow
           $env:LANGFLOW_SSRF_PROTECTION_ENABLED = "false"
           $env:LANGFLOW_SSRF_ALLOWED_HOSTS = "localhost,127.0.0.1"
           langflow run --host 127.0.0.1 --port 7860

    3) 在浏览器打开 http://localhost:7860 ，拖一个流程出来，然后：
           · 流程页 URL 里的 /flow/<这一串> 就是 flow_id
           · 右上角 Settings → API Keys 生成密钥，填进环境变量：
                 $env:LANGFLOW_FLOW_ID = "<你的 flow_id>"
                 $env:LANGFLOW_API_KEY = "YOUR_LANGFLOW_API_KEY"
           或者直接用命令行参数：--flow-id <ID> --api-key <KEY>

    4) 服务起来后重跑本文件，就会走真实调用分支。

    详细部署命令（含生产环境 docker compose 方式）见同目录的
    02_平台对比与部署_jxsd.py。
```

三处值得停一下：

1. **`flow_id` 与 `api_key` 打印成了占位符**，而不是空串 ——
   因为 `main()` 里做了 `args.flow_id or PLACEHOLDER_FLOW_ID`。
   于是 dry-run 打出来的请求里，**哪里还没配一眼就能看见**；
2. **`api_key` 只显示首尾**：那行 `shown_key = api_key if api_key.startswith("YOUR_") else f"{api_key[:4]}...{api_key[-4:]}"`
   的意思是「占位符照原样显示（它本来就不是密钥），真密钥只露首尾 4 位」。
   这是**防止密钥进日志**的最小改动；
3. **`session_id` 每次不同**（随机 uuid4），对照时看形状即可。

### 3.7 换个参数再跑一遍（走 `--dry-run` 显式分支）

上一格是「服务不可达 → 自动降级」，这一格是「服务可达我也会先看请求长什么样 → 显式 `--dry-run`」。
两种情况的输出**只差最后一句提示**：
自动降级会打印那段中文排障指引，显式 dry-run 不会。

这里用的是前面定义的 `run_main()`，所以它演示的正是
「notebook 里怎么用命令行参数跑同一个 `main()`」。

In [ ]:
run_main(["--dry-run", "--input", "帮我总结这篇论文", "--output-type", "text", "--timeout", "30"])

### 预期输出

```text
==============================================================================
2. 参数来源与本次调用的目标
==============================================================================
    base_url    : http://localhost:7860      （--base-url / LANGFLOW_BASE_URL）
    flow_id     : YOUR_FLOW_ID      （--flow-id / LANGFLOW_FLOW_ID）
    api_key     : YOUR_LANGFLOW_API_KEY      （--api-key / LANGFLOW_API_KEY，只显示首尾）
    input_value : 帮我总结这篇论文
    output_type : text
    dry_run     : True

==============================================================================
3. 健康探测：先确认服务在不在
==============================================================================
    GET http://localhost:7860/health → 连接被拒绝（服务没起来）
    → 服务不可达，降级为 dry-run

==============================================================================
4. DRY-RUN：只打印将要发送的请求，不真的发出去
==============================================================================
  ── 请求预览 ──
    POST http://localhost:7860/api/v1/run/YOUR_FLOW_ID
    Header: x-api-key: YOUR_LANGFLOW_API_KEY
    Header: Content-Type: application/json
    Body（JSON，中文不转义，方便肉眼核对）：
      {
            "output_type": "text",
            "input_type": "chat",
            "input_value": "帮我总结这篇论文",
            "session_id": "35e1223e-5a34-4d94-ad8c-9bf123cd3c75"
      }
    （--dry-run 指定，跳过真实请求。要去掉这个参数并确保 Langflow 已启动。）
```

**`output_type` 从 `chat` 变成 `text` 了** —— 这就是课案那三个输入框的实际效果：
同样一条流程，要「对话式返回」还是「纯文本返回」由**调用方**决定，不用改流程。

> 顺带注意：`--timeout 30` 在输出里看不到 —— 它不参与打印，
> 只在真实 POST 时进 `requests.request(..., timeout=args.timeout)`。

### 3.8 服务可达时的真实调用分支（本机走不到，🔴 需外部服务）

上面 `main()` 的**分支 B** 在本机永远不会执行 —— 因为 7860 没人听。
它需要的东西，逐项列在这里：

| 需要什么 | 怎么给 | 本机状态 |
|---|---|---|
| 一个跑着的 Langflow | `langflow run --host 127.0.0.1 --port 7860`（见 2.6 节） | ❌ 未启动 |
| `LANGFLOW_BASE_URL`（或 `--base-url`） | `$env:LANGFLOW_BASE_URL = "http://localhost:7860"` | ⚪ 未设（有默认值） |
| `LANGFLOW_FLOW_ID`（或 `--flow-id`） | 流程页 URL 里的 `/flow/<这一段>` | ❌ 未设（走占位符） |
| `LANGFLOW_API_KEY`（或 `--api-key`） | 界面右上角 Settings → API Keys 生成 | ❌ 未设（走占位符） |

三项配齐、服务起来之后，**重跑 3.6 那一格**（`run_main([])`）就会走分支 B，
输出会变成下面这样：

```text
==============================================================================
3. 健康探测：先确认服务在不在
==============================================================================
    GET http://localhost:7860/health → HTTP 200
    → 服务可达，继续真实调用

==============================================================================
4. 真实调用：POST http://localhost:7860/api/v1/run/<你的 flow_id>
==============================================================================
  ── 请求预览 ──
    POST http://localhost:7860/api/v1/run/<你的 flow_id>
    Header: x-api-key: sk-1...9abc
    Header: Content-Type: application/json
    Body（JSON，中文不转义，方便肉眼核对）：
      { ... }

  ✅ HTTP 200
  ── 原始响应体（课案就是 print(response.text)）──
  | {"session_id":"...","outputs":[{"inputs":{...},"outputs":[{"results":{"message":{"text":"..."}}}]}]}
  ── 试着直接取出回答文本 ──
  <流程真正吐出来的那句话>
```

分支 B 里有**两段值得单独记住**的代码：

**① 按状态码给不同提示**（而不是笼统一句「请求失败」）：

| 状态码 | 含义 | 代码给的提示 |
|---|---|---|
| 401 / 403 | 密钥不对或没传 | 检查 `--api-key` / `LANGFLOW_API_KEY` |
| 404 | `flow_id` 不存在 | 检查 `--flow-id` / `LANGFLOW_FLOW_ID` |

**② 从嵌套 JSON 里把回答文本捞出来**：Langflow 的返回是四层嵌套，
真正的话术藏在 `outputs[0].outputs[0].outputs[0].results.message.text`。
代码里那句连续 `.get(...)` **每一层都写了默认值 `[{}]` / `{}`**，
所以流程的返回结构不一样时不会 `KeyError` / `IndexError`，
而是走到最后那句「（这个流程的返回结构不是标准 chat 输出，请看上面的原始响应体）」。

> 这就是「教学脚本的失败提示本身也是教学内容」：宁可多写两个默认值，
> 也不要让学员在 JSON 里翻半天才发现是结构对不上。

## 小结

- **可视化平台不是「不用代码」，而是「把代码换成节点」**：
  四家里只有 Coze 闭源，私有化部署这一条基本就决定了选型；
- **抽象层次决定代价**：Dify 换来 RAG 开箱即用、n8n 换来几百个集成节点、
  Langflow 换来对 LangChain 组件的细粒度控制 —— **代价是你要懂 LangChain**；
- **流程搭完之后一定回到代码**：`POST /api/v1/run/<flow_id>` 加四个字段
  （`output_type` / `input_type` / `input_value` / `session_id`）就是全部；
- **工程化的三件事**，源文件逐个补上了课案的三个坑：

| 坑 | 补法 | 本 notebook 哪一格 |
|---|---|---|
| 密钥硬编码 | 命令行 > 环境变量 > 占位符 | 3.2 `parse_args()` |
| `flow_id` 硬编码 | 同上 | 3.2 / 3.6 |
| 连接失败就崩 | 发请求前先 `GET /health`，失败降级为 dry-run | 3.4 `probe_health()` / 3.6 `main()` |

- **`session_id` 决定记不记得上文**：复用同一个就是多轮会话，每次 `uuid4()` 就是单轮问答；
- **密钥只显示首尾**（`f"{api_key[:4]}...{api_key[-4:]}"`）是防止口令进日志的最小改动。

## 常见坑

1. **percent 源码里想显示一个真正的 `#`，要写两个 `#`。** markdown cell 的每一行都是一个
   `#` 注释，解析时只去掉「**一个** `#` + 一个空格」，所以源码里要写成
   「井号 空格 井号 空格 正文」，渲染出来才是「井号 空格 正文」，也就是那一行真的以 `#` 开头。
   ⚠️ 不要用 Markdown 的反斜杠转义（反斜杠 + 井号）去凑 —— 在代码围栏里反斜杠是**字面量**，
   那样会原样显示成「反斜杠 + 井号」，多出一个反斜杠。本节「预期输出」里那几行
   `# 创建 Conda 环境` 之类，源码里都是「井号 空格 井号 空格 ...」这个写法。
2. **`parser.parse_args()` 在内核里会 `SystemExit`。** Jupyter 的 `sys.argv` 是内核启动参数，
   argparse 不认。要么像 3.2 节那样临时替换 `sys.argv`，要么给 `parse_args()` 传 `argv` 列表。
3. **`requests` 的 except 顺序**：`ConnectionError` / `Timeout` 必须写在 `RequestException`
   **前面**，否则永远被父类吃掉，报错信息从「连接被拒绝」退化成「RequestException」。
4. **`base_url` 末尾的斜杠**：不 `rstrip("/")` 会拼出 `//health`、`//api/v1/run`。
   大多数服务器对 `//` 宽容，但没必要赌。
5. **`json.dumps` 默认会转义中文**（`ensure_ascii=True`），打印中文请求体时一定要传
   `ensure_ascii=False`，否则「橘醒时代」变成 `\u6a58\u9192\u65f6\u4ee3`。
6. **`docker` 命令在 ≠ Docker 守护进程在跑。** 只用 `shutil.which("docker")` 判断，
   在 Docker Desktop 没启动时会误报「环境就绪」；必须再跑一次 `docker info`。
7. **`--rm` 与数据持久化不冲突，但别用在生产**：数据在命名卷里，
   容器删了卷还在。生产环境去掉 `--rm`（见 2.4 节课案那段）。
8. **课案原文里 `ollama tun qwen3` 是笔误**（应为 `ollama run qwen3`）。
   本 notebook 保留原文以便对照，没有替课案改字 —— 遇到时按 `run` 执行。

## 官方链接

- 工作流与 Agent（本课「代码编排」那一半的官方说明）：<https://docs.langchain.com/oss/python/langgraph/workflows-agents>
- Langflow 官方文档（flow 的搭法与 API）：<https://docs.langflow.org/>
- Langflow GitHub（生产部署用的 `docker_example` 就在这里）：<https://github.com/langflow-ai/langflow>
- Dify 官方文档：<https://docs.dify.ai/>
- n8n 官方文档：<https://docs.n8n.io/>
- Coze 国内站：<https://code.coze.cn/home>
- arxiv API（Dify 那个检索工具用的接口）：<https://info.arxiv.org/help/api/index.html>
- Ollama 模型库（`bge-m3` 等）：<https://ollama.com/library>